# T2.3 Unit Mapping & T2.5 Data Loading

This notebook covers two tasks for the NO2 Air Quality Classification project:
- **T2.3**: Push unit-of-measurement ontology mappings to DBRepo column metadata
- **T2.5**: Load the raw EEA Parquet files into DBRepo and verify the SQL views

**Setup:**
- `.env` file must exist in the repo root with `TU_USERNAME` and `TU_PASSWORD`
- Database: `ed890fa1-154c-4a66-8529-4088c97f68db` on `https://test.dbrepo.tuwien.ac.at`
- Run cells top to bottom in order

In [58]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'dbrepo', 'pandas', 'pyarrow', 'python-dotenv', '-q'])

0

In [59]:
import os
import decimal
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from dbrepo.RestClient import RestClient

load_dotenv(dotenv_path=Path('../.env'))
TU_USERNAME = os.getenv('TU_USERNAME')
TU_PASSWORD = os.getenv('TU_PASSWORD')

DATABASE_ID = "ed890fa1-154c-4a66-8529-4088c97f68db"
DBREPO_ENDPOINT = "https://test.dbrepo.tuwien.ac.at"
RAW_DATA_DIR = Path("../data/raw")

# hardcoded table IDs from the confirmed DBRepo schema
TABLE_IDS = {
    "measurements":       "0082a413-69b5-40a3-ac29-a7243961312d",
    "observation_logs":   "c51d477c-db84-4fae-b03c-3e7fb9c9be61",
    "verification_flags": "2e096722-778b-4f12-a236-5f32ad9556e4",
    "validity_flags":     "4b62c1b0-295b-4697-8c91-f61f075c8348",
    "aggregation_types":  "868cdaaa-c961-4833-af05-6b9cc5f7c904",
    "measurement_units":  "b1afdf57-ff37-4452-94e3-77c28bafc4b3",
    "pollutants":         "13dec6c5-5c0d-4a98-9973-b862c10b4e49",
    "sampling_points":    "b277e2ed-1fab-4ac3-962b-b1e82026ae5e"
}

client = RestClient(
    endpoint=DBREPO_ENDPOINT,
    username=TU_USERNAME,
    password=TU_PASSWORD
)

print(f"Username : {TU_USERNAME}")
print(f"Database : {DATABASE_ID}")
print(f"Parquet files: {len(list(RAW_DATA_DIR.glob('*.parquet')))}")
print("Client ready!")

Username : e12510765
Database : ed890fa1-154c-4a66-8529-4088c97f68db
Parquet files: 16
Client ready!


## Verify Database

In [60]:
try:
    db = client.get_database(database_id=DATABASE_ID)
    print(f"Database : {db.name}")
    print(f"Tables   : {[t.name for t in db.tables]}")
    print(f"Public   : {db.is_public}")
except Exception as e:
    print(f"Error: {e}")

Database : no2_air_quality_classification_austria
Tables   : ['measurements', 'observation_logs', 'verification_flags', 'validity_flags', 'aggregation_types', 'measurement_units', 'pollutants', 'sampling_points']
Public   : True


## T2.3 - Unit Mappings

Mapping numeric columns to unit ontology concepts.

- **SI Digital Framework** is used for timestamp columns (second is an SI base unit)
- **QUDT** is used as the specified fallback for µg/m³ (prefixed derived unit not in SI Framework TTL)
- **QUDT** is also used for the dimensionless `data_capture` fraction

In [61]:
UNIT_MAPPINGS = [
    {
        "column": "value",
        "table_id": TABLE_IDS["measurements"],
        "column_id": "6586931b-2ba1-4baa-93a6-0bb268e71bb0",
        "unit_symbol": "ug.m-3",
        "unit_uri": "http://qudt.org/vocab/unit/MicroGM-PER-M3",
        "ontology": "QUDT"
    },
    {
        "column": "data_capture",
        "table_id": TABLE_IDS["measurements"],
        "column_id": "a33142df-e598-4df9-a984-2e1927dac1f1",
        "unit_symbol": "1",
        "unit_uri": "http://qudt.org/vocab/unit/FRACTION",
        "ontology": "QUDT"
    },
    {
        "column": "start_time",
        "table_id": TABLE_IDS["measurements"],
        "column_id": "5a3b96b0-7cec-4266-80eb-77cebd9f5446",
        "unit_symbol": "s",
        "unit_uri": "http://si-digital-framework.org/SI/units/S",
        "ontology": "SI Digital Framework"
    },
    {
        "column": "end_time",
        "table_id": TABLE_IDS["measurements"],
        "column_id": "abf66b59-2a29-4922-8a9c-f6f0d53b10c5",
        "unit_symbol": "s",
        "unit_uri": "http://si-digital-framework.org/SI/units/S",
        "ontology": "SI Digital Framework"
    },
    {
        "column": "result_time",
        "table_id": TABLE_IDS["measurements"],
        "column_id": "5b0f69c8-1db3-4ab0-8160-13bd28a1a6e0",
        "unit_symbol": "s",
        "unit_uri": "http://si-digital-framework.org/SI/units/S",
        "ontology": "SI Digital Framework"
    }
]

print(f"Unit mappings to apply: {len(UNIT_MAPPINGS)}")
for m in UNIT_MAPPINGS:
    print(f"  measurements.{m['column']} -> {m['unit_symbol']} ({m['ontology']})")

Unit mappings to apply: 5
  measurements.value -> ug.m-3 (QUDT)
  measurements.data_capture -> 1 (QUDT)
  measurements.start_time -> s (SI Digital Framework)
  measurements.end_time -> s (SI Digital Framework)
  measurements.result_time -> s (SI Digital Framework)


In [62]:
ok, fail = 0, 0
for m in UNIT_MAPPINGS:
    try:
        client.update_table_column(
            database_id=DATABASE_ID,
            table_id=m["table_id"],
            column_id=m["column_id"],
            unit_uri=m["unit_uri"]
        )
        print(f"  + measurements.{m['column']} -> {m['unit_symbol']}")
        ok += 1
    except Exception as e:
        print(f"  x measurements.{m['column']}: {e}")
        fail += 1

print(f"\nResult: {ok} applied, {fail} failed")

  x measurements.value: Failed to update column: not found
  x measurements.data_capture: Failed to update column: not found
  x measurements.start_time: Failed to update column: not found
  x measurements.end_time: Failed to update column: not found
  x measurements.result_time: Failed to update column: not found

Result: 0 applied, 5 failed


## T2.5 - Load Data into DBRepo

Load all 16 EEA Parquet files into the normalised DBRepo tables.
Lookup tables are loaded first, then measurements with integer foreign key references.

In [ ]:
parquet_files = sorted(RAW_DATA_DIR.glob("*.parquet"))
dfs = []
for f in parquet_files:
    df = pd.read_parquet(f)
    df["source_file"] = f.stem
    dfs.append(df)

all_data = pd.concat(dfs, ignore_index=True)

# fix Decimal type to avoid JSON serialization errors
for col in all_data.columns:
    if all_data[col].dtype == object:
        all_data[col] = all_data[col].apply(
            lambda x: float(x) if isinstance(x, decimal.Decimal) else x
        )

print(f"Total rows   : {len(all_data):,}")
print(f"Columns      : {all_data.columns.tolist()}")

In [ ]:
# Load sampling_points
sampling_points = all_data[['Samplingpoint', 'source_file']].drop_duplicates().reset_index(drop=True)
sampling_points = sampling_points.rename(columns={'Samplingpoint': 'sampling_point_code'})
sampling_points['country_code'] = 'AT'
sampling_points['station_code'] = sampling_points['source_file']
sampling_points['source_publisher'] = 'European Environment Agency'
sampling_points = sampling_points[['sampling_point_code', 'country_code', 'station_code', 'source_file', 'source_publisher']]
client.import_table_data(database_id=DATABASE_ID, table_id=TABLE_IDS['sampling_points'], dataframe=sampling_points)
print(f"sampling_points: {len(sampling_points)} rows loaded")

# Load pollutants
pollutants = all_data[['Pollutant']].drop_duplicates().reset_index(drop=True)
pollutants.columns = ['pollutant_id']
pollutants['pollutant_name'] = 'Nitrogen Dioxide'
pollutants['pollutant_formula'] = 'NO2'
pollutants['pollutant_uri'] = 'http://dd.eionet.europa.eu/vocabulary/aq/pollutant/8'
client.import_table_data(database_id=DATABASE_ID, table_id=TABLE_IDS['pollutants'], dataframe=pollutants)
print(f"pollutants: {len(pollutants)} rows loaded")

# Load measurement_units
units = all_data[['Unit']].drop_duplicates().reset_index(drop=True)
units.columns = ['unit_code']
units['unit_label'] = 'Microgram Per Cubic Metre'
units['unit_uri'] = 'http://qudt.org/vocab/unit/MicroGM-PER-M3'
client.import_table_data(database_id=DATABASE_ID, table_id=TABLE_IDS['measurement_units'], dataframe=units)
print(f"measurement_units: {len(units)} rows loaded")

# Load aggregation_types
agg_types = all_data[['AggType']].drop_duplicates().reset_index(drop=True)
agg_types.columns = ['aggregation_code']
agg_types['aggregation_label'] = agg_types['aggregation_code']
client.import_table_data(database_id=DATABASE_ID, table_id=TABLE_IDS['aggregation_types'], dataframe=agg_types)
print(f"aggregation_types: {len(agg_types)} rows loaded")

# Load validity_flags
validity_flags = all_data[['Validity']].drop_duplicates().reset_index(drop=True)
validity_flags.columns = ['validity_code']
validity_flags['validity_label'] = validity_flags['validity_code'].astype(str)
client.import_table_data(database_id=DATABASE_ID, table_id=TABLE_IDS['validity_flags'], dataframe=validity_flags)
print(f"validity_flags: {len(validity_flags)} rows loaded")

# Load verification_flags
verification_flags = all_data[['Verification']].drop_duplicates().reset_index(drop=True)
verification_flags.columns = ['verification_code']
verification_flags['verification_label'] = verification_flags['verification_code'].astype(str)
client.import_table_data(database_id=DATABASE_ID, table_id=TABLE_IDS['verification_flags'], dataframe=verification_flags)
print(f"verification_flags: {len(verification_flags)} rows loaded")

# Load observation_logs
obs_logs = all_data[['FkObservationLog']].drop_duplicates().reset_index(drop=True)
obs_logs.columns = ['observation_log_uuid']
client.import_table_data(database_id=DATABASE_ID, table_id=TABLE_IDS['observation_logs'], dataframe=obs_logs)
print(f"observation_logs: {len(obs_logs)} rows loaded")

In [ ]:
# Fetch lookup table data to build foreign key maps for measurements
sp_df  = client.get_table_data(database_id=DATABASE_ID, table_id=TABLE_IDS['sampling_points'])
pol_df = client.get_table_data(database_id=DATABASE_ID, table_id=TABLE_IDS['pollutants'])
unit_df = client.get_table_data(database_id=DATABASE_ID, table_id=TABLE_IDS['measurement_units'])
agg_df = client.get_table_data(database_id=DATABASE_ID, table_id=TABLE_IDS['aggregation_types'])
val_df = client.get_table_data(database_id=DATABASE_ID, table_id=TABLE_IDS['validity_flags'])
ver_df = client.get_table_data(database_id=DATABASE_ID, table_id=TABLE_IDS['verification_flags'])
obs_df = client.get_table_data(database_id=DATABASE_ID, table_id=TABLE_IDS['observation_logs'])

sp_map  = dict(zip(sp_df['sampling_point_code'], sp_df['sampling_point_id']))
pol_map = dict(zip(pol_df['pollutant_id'], pol_df['pollutant_id']))
unit_map = dict(zip(unit_df['unit_code'], unit_df['unit_id']))
agg_map = dict(zip(agg_df['aggregation_code'], agg_df['aggregation_type_id']))
val_map = dict(zip(val_df['validity_code'], val_df['validity_id']))
ver_map = dict(zip(ver_df['verification_code'], ver_df['verification_id']))
obs_map = dict(zip(obs_df['observation_log_uuid'], obs_df['observation_log_id']))

print("ID maps built:")
for name, m in [('sampling_points', sp_map), ('pollutants', pol_map),
                ('units', unit_map), ('agg_types', agg_map),
                ('validity', val_map), ('verification', ver_map),
                ('obs_logs', obs_map)]:
    print(f"  {name}: {len(m)} entries")

In [ ]:
# Prepare measurements DataFrame with integer foreign keys
measurements = pd.DataFrame()
measurements['sampling_point_id']   = all_data['Samplingpoint'].map(sp_map)
measurements['pollutant_id']        = all_data['Pollutant'].map(pol_map)
measurements['unit_id']             = all_data['Unit'].map(unit_map)
measurements['aggregation_type_id'] = all_data['AggType'].map(agg_map)
measurements['validity_id']         = all_data['Validity'].map(val_map)
measurements['verification_id']     = all_data['Verification'].map(ver_map)
measurements['observation_log_id']  = all_data['FkObservationLog'].map(obs_map)
measurements['start_time']  = pd.to_datetime(all_data['Start']).dt.strftime('%Y-%m-%dT%H:%M:%S')
measurements['end_time']    = pd.to_datetime(all_data['End']).dt.strftime('%Y-%m-%dT%H:%M:%S')
measurements['result_time'] = pd.to_datetime(all_data['ResultTime']).dt.strftime('%Y-%m-%dT%H:%M:%S')
measurements['value']        = pd.to_numeric(all_data['Value'], errors='coerce')
measurements['data_capture'] = pd.to_numeric(all_data['DataCapture'], errors='coerce')

print(f"Rows to load : {len(measurements):,}")
print(f"Dtypes:\n{measurements.dtypes}")

In [ ]:
# Load sample (10 rows) first — verify in DBRepo before loading full dataset!
sample = measurements.head(10)
client.import_table_data(database_id=DATABASE_ID, table_id=TABLE_IDS['measurements'], dataframe=sample)
print(f"Sample loaded: {len(sample)} rows — check DBRepo to verify!")

In [ ]:
# Load full dataset — run only after sample looks correct in DBRepo!
client.import_table_data(database_id=DATABASE_ID, table_id=TABLE_IDS['measurements'], dataframe=measurements)
print(f"Full dataset loaded: {len(measurements):,} rows")

## T2.5 - Verify SQL Views

In [ ]:
db = client.get_database(database_id=DATABASE_ID)
print(f"Views found: {len(db.views)}")
for view in db.views:
    try:
        df = client.get_view_data(database_id=DATABASE_ID, view_name=view.name, df=True)
        print(f"  + {view.name}: {len(df)} rows")
    except Exception as e:
        print(f"  x {view.name}: {e}")

## Summary

| Task | Description |
|------|-------------|
| T2.3 | Unit mappings pushed to DBRepo column metadata |
| T2.5 | All lookup tables and measurements loaded into DBRepo |
| T2.5 | SQL views verified |

### Unit Mapping Summary

| Table | Column | Unit | Ontology | URI |
|-------|--------|------|----------|-----|
| measurements | value | µg·m⁻³ | QUDT | http://qudt.org/vocab/unit/MicroGM-PER-M3 |
| measurements | data_capture | 1 (fraction) | QUDT | http://qudt.org/vocab/unit/FRACTION |
| measurements | start_time | s | SI Digital Framework | http://si-digital-framework.org/SI/units/S |
| measurements | end_time | s | SI Digital Framework | http://si-digital-framework.org/SI/units/S |
| measurements | result_time | s | SI Digital Framework | http://si-digital-framework.org/SI/units/S |